#Transformar Dados de Sprints

- 1.Ler a tabela bronze sprints
- 2.Manter apenas as colunas necessárias para análise (Remover a coluna url)
- 3.Padronizar os nomes das colunas usando snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
- 4.Renomear colunas para torná-las mais significativas (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
- 5.Filtrar linhas onde season, round, custructor_id ou driver_id estejam nulos (validação de chave de negócio)
- 6.Remover registros duplicados
- 7.Transformar os valores da coluna race_name para Title Case (Primeira Letra Maiúscula)
- 8.Escrever os dados transformados na tabela silver sprints

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = (
    spark.table(bronze_table)
    .drop("url")
    .withColumnsRenamed({
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName": "race_name",
        "positionText": "finish_position_text",
        "date": "race_date",
        "grid": "grid_position",
        "laps": "completed_laps",
        "number": "car_number",
        "position": "finish_position"
    })
)

In [0]:
sprints_null_df = (
    sprints_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull ()
    )
    .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)


In [0]:
display(sprints_df.count() - sprints_null_df.count())

20

In [0]:
sprints_final_df = (
    sprints_null_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,ingestion_timestamp,source_file
2021-07-18,British Grand Prix,10,2021,red_bull,max_verstappen,2,17,33,3.0,1,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,hamilton,1,17,44,2.0,2,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,bottas,3,17,77,1.0,3,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,ferrari,leclerc,4,17,16,0.0,4,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mclaren,norris,6,17,4,0.0,5,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mclaren,ricciardo,7,17,3,0.0,6,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alpine,alonso,11,17,14,0.0,7,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,aston_martin,vettel,10,17,5,0.0,8,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,williams,russell,8,17,63,0.0,9,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alpine,ocon,13,17,31,0.0,10,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json


In [0]:
(
    sprints_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,ingestion_timestamp,source_file
2021-07-18,British Grand Prix,10,2021,aston_martin,stroll,15,17,18,0.0,14,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2021-11-14,São Paulo Grand Prix,19,2021,williams,latifi,16,24,6,0.0,16,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2021.json
2022-07-10,Austrian Grand Prix,11,2022,mercedes,hamilton,9,23,44,1.0,8,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2022.json
2023-04-30,Azerbaijan Grand Prix,4,2023,red_bull,max_verstappen,3,17,1,6.0,3,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
2023-04-30,Azerbaijan Grand Prix,4,2023,williams,albon,7,17,23,0.0,9,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
2023-04-30,Azerbaijan Grand Prix,4,2023,alfa,zhou,14,17,24,0.0,12,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
2023-04-30,Azerbaijan Grand Prix,4,2023,alphatauri,tsunoda,16,2,22,0.0,19,null,Collision damage,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
2023-07-02,Austrian Grand Prix,9,2023,ferrari,leclerc,9,24,16,0.0,12,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
2023-07-30,Belgian Grand Prix,12,2023,aston_martin,stroll,14,11,18,0.0,11,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
2023-10-22,United States Grand Prix,18,2023,haas,hulkenberg,16,19,27,0.0,15,null,Finished,2026-08-15T19:18:57.834Z,dbfs:/Volumes/formula1/landing/arquivos/sprints/sprints_2023.json
